# Flipkart Gridlock 2.0: V9 Pure Raw Architecture
## Native Categorical Embedding (CatBoost)

**Architectural Paradigm:**
This pipeline discards manual spatiotemporal feature engineering. It feeds the unmanipulated, raw dataset directly into a symmetrical Oblivious Tree framework (CatBoost). The model utilizes internal categorical embeddings to autonomously map spatial networks and temporal cycles without external trigonometric or lag-based processing.

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, Pool
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold

warnings.filterwarnings('ignore')
np.random.seed(42)

data_paths = [".", "data/raw", "../../data/raw"]
base_path = next((path for path in data_paths if os.path.exists(os.path.join(path, "train.csv"))), None)

raw_train = pd.read_csv(os.path.join(base_path, "train.csv"))
raw_test = pd.read_csv(os.path.join(base_path, "test.csv"))

y_train = raw_train['demand'].values
submission_index = raw_test['Index'].values

In [4]:
def parse_raw_matrix(df):
    df_raw = df.copy()
    
    time_split = df_raw['timestamp'].str.split(':', expand=True)
    df_raw['raw_hour'] = time_split[0]
    df_raw['raw_minute'] = time_split[1]
    
    df_raw.drop(columns=['timestamp'], inplace=True, errors='ignore')

    categorical_columns = ['Weather', 'RoadType', 'LargeVehicles', 'Landmarks']
    for col in categorical_columns:
        if col in df_raw.columns:
            df_raw[col] = df_raw[col].fillna('Missing')
        
    if 'Temperature' in df_raw.columns:
        df_raw['Temperature'] = df_raw['Temperature'].fillna(-999.0)
    
    cat_features_list = ['geohash', 'raw_hour', 'raw_minute', 'day'] + categorical_columns

    cat_features_list = [c for c in cat_features_list if c in df_raw.columns]
    
    for col in cat_features_list:
        df_raw[col] = df_raw[col].astype(str)
        
    return df_raw, cat_features_list

X_train_raw, categorical_features = parse_raw_matrix(raw_train.drop(columns=['demand', 'Index'], errors='ignore'))
X_test_raw, _ = parse_raw_matrix(raw_test.drop(columns=['Index'], errors='ignore'))

X_test_raw = X_test_raw[X_train_raw.columns]

print(f"Matrix parsing complete. Expected features: {list(X_train_raw.columns)}")

Matrix parsing complete. Expected features: ['geohash', 'day', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather', 'raw_hour', 'raw_minute']


In [5]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# CatBoost parameters optimized for deep categorical extraction
cat_params = {
    'iterations': 3500,
    'learning_rate': 0.03,
    'depth': 8,
    'loss_function': 'RMSE',
    'eval_metric': 'R2',
    'random_seed': 42,
    'task_type': 'CPU',
    'early_stopping_rounds': 150,
    'verbose': 100
}

oof_predictions = np.zeros(len(X_train_raw))
test_predictions = np.zeros(len(X_test_raw))

print("Executing K-Fold Native Categorical Training...")

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_raw)):
    X_tr, y_tr = X_train_raw.iloc[train_idx], y_train[train_idx]
    X_va, y_va = X_train_raw.iloc[val_idx], y_train[val_idx]
    
    train_pool = Pool(X_tr, y_tr, cat_features=categorical_features)
    val_pool = Pool(X_va, y_va, cat_features=categorical_features)
    test_pool = Pool(X_test_raw, cat_features=categorical_features)
    
    model = CatBoostRegressor(**cat_params)
    model.fit(train_pool, eval_set=val_pool)
    
    oof_predictions[val_idx] = model.predict(val_pool)
    test_predictions += model.predict(test_pool) / kf.n_splits
    
    fold_r2 = max(0, 100 * r2_score(y_va, oof_predictions[val_idx]))
    print(f"Fold {fold+1} Completed. R2 Score: {fold_r2:.4f}\n")

Executing K-Fold Native Categorical Training...
0:	learn: 0.0439289	test: 0.0448300	best: 0.0448300 (0)	total: 70.5ms	remaining: 4m 6s
100:	learn: 0.8865055	test: 0.8927141	best: 0.8927141 (100)	total: 876ms	remaining: 29.5s
200:	learn: 0.9061909	test: 0.9130054	best: 0.9130054 (200)	total: 1.77s	remaining: 29s
300:	learn: 0.9141061	test: 0.9190614	best: 0.9190614 (300)	total: 2.77s	remaining: 29.4s
400:	learn: 0.9184058	test: 0.9219043	best: 0.9219043 (400)	total: 3.7s	remaining: 28.6s
500:	learn: 0.9216177	test: 0.9237663	best: 0.9237663 (500)	total: 4.68s	remaining: 28s
600:	learn: 0.9245124	test: 0.9253066	best: 0.9253066 (600)	total: 5.69s	remaining: 27.4s
700:	learn: 0.9276698	test: 0.9270572	best: 0.9270572 (700)	total: 6.7s	remaining: 26.7s
800:	learn: 0.9295415	test: 0.9277605	best: 0.9277699 (799)	total: 7.67s	remaining: 25.8s
900:	learn: 0.9312729	test: 0.9285652	best: 0.9285825 (898)	total: 8.68s	remaining: 25s
1000:	learn: 0.9331119	test: 0.9292445	best: 0.9292806 (984)	to

In [6]:
final_r2 = max(0, 100 * r2_score(y_train, oof_predictions))

# Enforce physical capacity boundaries
final_test_predictions = np.clip(test_predictions, 0.0, 1.0)

submission_df = pd.DataFrame({
    'Index': submission_index,
    'demand': final_test_predictions
})

submission_df.to_csv("submission_v11.csv", index=False)

print("Pipeline execution complete.")
print(f"OOF Terminal R2 Score: {final_r2:.4f}")
print("Output generated: submission_v11.csv")

Pipeline execution complete.
OOF Terminal R2 Score: 93.4306
Output generated: submission_v11.csv
